
# Ames Housing: Cross-Validation

This notebook demonstrates how **cross-validation** can be used to obtain a more reliable estimate of model performance, especially when the available dataset is small.

We focus on:

- a single train/validation split,
- **k-fold cross-validation**,
- **leave-one-out cross-validation (LOOCV)** on a small subset,
- why cross-validation is useful for model comparison,
- and why an independent test set should still be kept for final evaluation.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    KFold,
    LeaveOneOut,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42



## 1. Load the Ames Housing dataset


In [ ]:

ames = fetch_openml(name="house_prices", as_frame=True)
df = ames.frame.copy()

df["SalePrice"] = pd.to_numeric(df["SalePrice"])

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt"
]

X = df[features].copy()
y = df["SalePrice"].copy()

print("Dataset size:", X.shape)
X.head()



## 2. Keep an independent test set

Cross-validation will be used only on the development data.

The test set is held aside and is not used to choose the model.


In [ ]:

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Development set:", len(X_dev))
print("Test set       :", len(X_test))



## 3. Why can a single validation split be unstable?

With a small dataset, model performance may depend strongly on which observations happen to be placed in the validation set.

To illustrate this, we create several different train/validation splits and evaluate the same model.


In [ ]:

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

single_split_results = []

for seed in range(10):
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev,
        test_size=0.20,
        random_state=seed
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = mean_squared_error(y_val, pred) ** 0.5

    single_split_results.append(rmse)

single_split_results = np.array(single_split_results)

print(f"Mean RMSE across splits: ${single_split_results.mean():,.0f}")
print(f"Lowest RMSE            : ${single_split_results.min():,.0f}")
print(f"Highest RMSE           : ${single_split_results.max():,.0f}")


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), single_split_results, marker="o")
plt.xlabel("Random train/validation split")
plt.ylabel("Validation RMSE ($)")
plt.title("The evaluation changes with the data split")
plt.tight_layout()
plt.show()



## 4. k-Fold cross-validation

In **k-fold cross-validation**, the development dataset is divided into \(k\) folds.

For each iteration:

- one fold is used for validation,
- the remaining \(k-1\) folds are used for training.

Every observation is therefore used for validation exactly once.

The final score is typically the average across all folds:

\[
\text{CV score} = \frac{1}{k}\sum_{i=1}^{k}\text{score}_i
\]


In [ ]:

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# sklearn returns negative MSE because larger scores are conventionally "better".
neg_mse_scores = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=kf,
    scoring="neg_mean_squared_error"
)

fold_rmse = np.sqrt(-neg_mse_scores)

for i, score in enumerate(fold_rmse, start=1):
    print(f"Fold {i}: RMSE = ${score:,.0f}")

print()
print(f"Mean 5-fold CV RMSE: ${fold_rmse.mean():,.0f}")
print(f"Std. deviation      : ${fold_rmse.std():,.0f}")


In [ ]:

plt.figure(figsize=(7, 4))
plt.bar(range(1, 6), fold_rmse)
plt.xlabel("Fold")
plt.ylabel("RMSE ($)")
plt.title("5-fold cross-validation results")
plt.xticks(range(1, 6))
plt.tight_layout()
plt.show()



## 5. Comparing models with cross-validation

Cross-validation is especially useful when we want to compare several model choices without relying on one lucky or unlucky validation split.

Here we compare ordinary linear regression with several Ridge regression strengths.


In [ ]:

candidates = {
    "Linear Regression": LinearRegression(),
    "Ridge alpha=0.1": Ridge(alpha=0.1),
    "Ridge alpha=1": Ridge(alpha=1.0),
    "Ridge alpha=10": Ridge(alpha=10.0),
    "Ridge alpha=100": Ridge(alpha=100.0)
}

cv_results = []

for name, estimator in candidates.items():
    candidate_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    scores = cross_val_score(
        candidate_pipeline,
        X_dev,
        y_dev,
        cv=kf,
        scoring="neg_mean_squared_error"
    )

    rmse_scores = np.sqrt(-scores)

    cv_results.append({
        "Model": name,
        "Mean CV RMSE": rmse_scores.mean(),
        "Std CV RMSE": rmse_scores.std()
    })

cv_results = pd.DataFrame(cv_results).sort_values("Mean CV RMSE")
cv_results



The model with the lowest average cross-validation RMSE is selected.

Notice that **the test set has still not been used**.



## 6. Leave-One-Out Cross-Validation

For very small datasets, we may want to use almost all available observations for training.

**Leave-One-Out Cross-Validation (LOOCV)** is the extreme case of k-fold cross-validation:

- train on \(n-1\) observations,
- validate on the one remaining observation,
- repeat this \(n\) times.

LOOCV makes very efficient use of a small dataset, but it is computationally expensive because the model must be fitted once for every observation.

To keep this demonstration fast, we apply LOOCV to a small subset of the Ames development data.


In [ ]:

small_X = X_dev.sample(n=120, random_state=RANDOM_STATE)
small_y = y_dev.loc[small_X.index]

loo = LeaveOneOut()

loo_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# MAE is convenient for LOOCV because each validation fold contains one observation.
loo_neg_mae = cross_val_score(
    loo_model,
    small_X,
    small_y,
    cv=loo,
    scoring="neg_mean_absolute_error"
)

loo_mae = -loo_neg_mae

print("Number of LOOCV fits:", len(loo_mae))
print(f"LOOCV mean MAE       : ${loo_mae.mean():,.0f}")



### Leave-\(p\)-Out / Leave-\(k\)-Out idea

A more general version is to leave out more than one observation at a time. This is usually called **Leave-\(p\)-Out Cross-Validation**.

It can be useful for extremely small datasets, but the number of possible combinations grows very quickly:

\[
\binom{n}{p}
\]

For this reason, full Leave-\(p\)-Out validation is rarely practical except for very small \(n\).



## 7. When should cross-validation be used?

Cross-validation is particularly useful when:

- the dataset is too small to comfortably reserve a large validation set;
- model performance changes substantially depending on the train/validation split;
- several models or hyperparameters must be compared;
- we want a more stable estimate of generalization performance.

In many practical problems, **5-fold or 10-fold cross-validation** provides a good compromise between reliability and computational cost.

LOOCV or Leave-\(p\)-Out approaches are mainly useful when the dataset is very small and repeated fitting is computationally manageable.



## 8. Final model evaluation on the untouched test set

After cross-validation has been used to select the model, we fit the selected model on the entire development set and evaluate it once on the test set.


In [ ]:

best_name = cv_results.iloc[0]["Model"]
print("Selected model:", best_name)

best_estimator = candidates[best_name]

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", best_estimator)
])

final_model.fit(X_dev, y_dev)
test_pred = final_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
test_r2 = r2_score(y_test, test_pred)

print(f"Test MAE : ${test_mae:,.0f}")
print(f"Test RMSE: ${test_rmse:,.0f}")
print(f"Test R²  : {test_r2:.3f}")



## Key takeaways

- A single validation split can give an unstable estimate when the dataset is small.
- **k-fold cross-validation** evaluates the model across several different train/validation splits.
- The average CV score is usually more reliable than one validation score.
- **5-fold and 10-fold cross-validation** are common practical choices.
- **LOOCV** uses almost all observations for training, but requires many model fits.
- Full Leave-\(p\)-Out validation becomes computationally expensive very quickly.
- Cross-validation is used for model selection; an independent **test set should ideally remain untouched until the end**.
